In [0]:
product_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(
        "abfss://bronze@adfassignment7.dfs.core.windows.net/sales_view/product/"
    )

display(product_df)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@adfassignment7.dfs.core.windows.net/sales_view/"
    )
)

In [0]:
product_bronze_path = "abfss://bronze@adfassignment7.dfs.core.windows.net/sales_view/products/"

In [0]:
product_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(product_bronze_path)

display(product_df)

In [0]:
print(product_df.columns)

In [0]:
# Step 2A: Create UDF function for snake_case

import re

def to_snake_case(column_name):
    column_name = re.sub(r'(?<!^)(?=[A-Z])', '_', column_name)
    column_name = column_name.replace(" ", "_")
    column_name = column_name.replace("-", "_")
    return column_name.lower()

In [0]:
# Step 2: Convert Product column headers to snake_case

for column in product_df.columns:
    product_df = product_df.withColumnRenamed(
        column,
        to_snake_case(column)
    )

print(product_df.columns)

In [0]:
# Step 3: Create sub_category

from pyspark.sql.functions import col, when

product_df = product_df.withColumn(
    "sub_category",
    when(col("category_id") == 1, "phone")
    .when(col("category_id") == 2, "laptop")
    .when(col("category_id") == 3, "playstation")
    .when(col("category_id") == 4, "e-device")
    .otherwise(None)
)

display(product_df)

In [0]:
# Step 5: Upsert Product data to Silver
from delta.tables import DeltaTable
silver_path = "abfss://silver@adfassignment7.dfs.core.windows.net/sales_view/product/"

if DeltaTable.isDeltaTable(spark, silver_path):

    target = DeltaTable.forPath(spark, silver_path)

    target.alias("target") \
        .merge(
            product_df.alias("source"),
            "target.product_id = source.product_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    print("Product data upserted successfully to Silver")

else:

    product_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(silver_path)

    print("Product Silver Delta table created successfully")